# 03 - AI re-coding of refund reasons

Problem: 42% of refunds are GW-OTHER (first option in the dropdown), 879 above the Rs 500 goodwill cap. Is that real goodwill over the cap, or a real reason (return, failed payment...) with a lazy code?

Plan:
1. pull a stratified sample -> **I label it by hand first** (gold set)
2. keyword baseline (no AI) -> what the AI has to beat
3. LLM with prompt v1, cached, scored on the gold set

In [1]:
import os, json, re
import pandas as pd

pd.set_option('display.max_colwidth', 120)

df = pd.read_csv('output/tickets_clean.csv')
refunds = df[df.refund_inr.notna()].copy()

CODES = ['GW-OTHER', 'DOA-REPL', 'LOST-TRANSIT', 'DUP-PAYMENT', 'CANCEL',
         'PRICE-ADJ', 'RETURN-QC-OK', 'WTY-BUYBACK']

print('refunds:', len(refunds), '| GW-OTHER:', (refunds.refund_reason_code == 'GW-OTHER').sum())

refunds: 2340 | GW-OTHER: 991


## 1. Gold set sample (60 tickets)

- 25 GW-OTHER above cap (the money question)
- 10 GW-OTHER at/below cap (probably real goodwill - checks the model doesn't re-code everything)
- 15 other codes (agent picked something specific - model should mostly agree)
- 10 refund + replacement

File is only written once so my labels never get overwritten.

In [2]:
GOLD_PATH = 'eval/gold_set.csv'

gw = refunds.refund_reason_code == 'GW-OTHER'
over = refunds.flag_gw_over_cap
both = refunds.flag_refund_and_repl

strata = [
    ('gw_over_cap', refunds[gw & over & ~both], 25),
    ('gw_under_cap', refunds[gw & ~over & ~both], 10),
    ('other_code', refunds[~gw & ~both], 15),
    ('refund_and_repl', refunds[both], 10),
]

if not os.path.exists(GOLD_PATH):
    gold = pd.concat([d.sample(n, random_state=42).assign(stratum=name) for name, d, n in strata])
    gold = gold[['ticket_id', 'stratum', 'category', 'refund_inr', 'refund_reason_code',
                 'replacement_issued', 'customer_message', 'agent_notes']]
    gold['my_code'] = ''          # fill by hand: one of CODES or UNCLEAR
    gold['my_real_goodwill'] = '' # Y if it's a genuine goodwill gesture, N otherwise
    gold.to_csv(GOLD_PATH, index=False)
    print('written', GOLD_PATH)

gold = pd.read_csv(GOLD_PATH, keep_default_na=False)
print(gold.stratum.value_counts())
print('labelled so far:', (gold.my_code != '').sum(), '/', len(gold))

stratum
gw_over_cap        25
other_code         15
gw_under_cap       10
refund_and_repl    10
Name: count, dtype: int64
labelled so far: 60 / 60


## 2. Keyword baseline

Simple rules on the agent note (+ message). First match wins, order matters.

In [3]:
RULES = [
    ('CANCEL',       r'cancel'),
    ('DUP-PAYMENT',  r'deduct|twice|double (charge|payment)|failed ord|without ord|utr|payment gateway'),
    ('LOST-TRANSIT', r'lost|not received|undeliver|never arrived'),
    ('DOA-REPL',     r'\bdoa\b|dead on arrival|dead|not (turning|switching) on'),
    ('WTY-BUYBACK',  r'buy.?back'),
    ('PRICE-ADJ',    r'price|coupon|discount'),
    ('RETURN-QC-OK', r'\bqc\b|reverse pick|return'),
    ('GW-OTHER',     r'goodwill|gesture|compensat'),
]

def keyword_code(note, msg):
    text = f'{note} {msg}'.lower()
    for code, pat in RULES:
        if re.search(pat, text):
            return code
    return 'UNCLEAR'

refunds['kw_code'] = [keyword_code(n, m) for n, m in zip(refunds.agent_notes, refunds.customer_message)]

In [4]:
# rough check before hand labels: on non-GW rows the agent chose a specific code, so use it as a proxy truth
specific = refunds[refunds.refund_reason_code != 'GW-OTHER']
print('keyword agrees with agent code on specific codes:', (specific.kw_code == specific.refund_reason_code).mean().round(3))
pd.crosstab(specific.refund_reason_code, specific.kw_code)

keyword agrees with agent code on specific codes: 0.523


kw_code,CANCEL,DOA-REPL,DUP-PAYMENT,GW-OTHER,LOST-TRANSIT,PRICE-ADJ,RETURN-QC-OK,UNCLEAR
refund_reason_code,,,,,,,,
CANCEL,216,0,0,0,0,0,0,6
DOA-REPL,0,2,8,7,7,2,19,102
DUP-PAYMENT,0,0,257,0,0,1,0,63
LOST-TRANSIT,0,0,0,0,10,2,1,34
PRICE-ADJ,0,0,0,0,0,64,0,7
RETURN-QC-OK,0,0,107,4,35,2,156,148
WTY-BUYBACK,0,2,4,7,5,1,12,58


In [5]:
# what the keyword baseline thinks the GW-OTHER refunds really are
g = refunds[refunds.refund_reason_code == 'GW-OTHER']
g.groupby('kw_code').refund_inr.agg(['count', 'sum']).sort_values('count', ascending=False)

,count,sum
kw_code,,
UNCLEAR,372,1130989.0
DUP-PAYMENT,236,638718.0
RETURN-QC-OK,129,381013.0
CANCEL,120,355854.0
LOST-TRANSIT,51,171666.0
PRICE-ADJ,41,81260.0
DOA-REPL,22,63192.0
GW-OTHER,20,84344.0


In [6]:
# where it breaks - read some
g[['ticket_id', 'kw_code', 'agent_notes']].sample(8, random_state=7)

,ticket_id,kw_code,agent_notes
2294,TK-242877,DUP-PAYMENT,1. failed order after payment\n2. conf with payment gateway\n3. refund of rs 1169 initiated to source
11164,TK-253989,UNCLEAR,"cx: rapid batery drain | advised disabling anc/aod to test -> advised settings change, cx satisfied.\nReversed the p..."
1877,TK-242358,UNCLEAR,resolved on call ~Manish
5667,TK-247128,UNCLEAR,customer reached out - unit rcvd damaged. chk w/ crr for damage claim. full rfnd processed (1105).
300,TK-240381,CANCEL,"cx reached out - cancellation request. Cancelled before dispatch. Already shipped, advised RTO/return. Cx satisfied ..."
10745,TK-253466,UNCLEAR,"rfnd approved, rs 712, 5-7 working days"
7747,TK-249730,UNCLEAR,"reset done, paired ok. issue: device not discoverable. asked cx to reset the device. issued refund + replacement bot..."
278,TK-240354,CANCEL,cx: cancel ord | chk shipment satus -> refunded rs 4248


## 3. LLM labelling (prompt v1)

Gemini 3.5 Flash-Lite - stable flash-lite has the biggest free quota, and this is a simple classification job. temperature 0, JSON output.

To go easy on the rate limit: 20 tickets per call, small pause between calls. Every answer is cached by ticket_id + prompt version, so re-running costs nothing and the final tool runs without a key.

In [7]:
from dotenv import load_dotenv
load_dotenv(override=True)   # .env key wins over any old system env var

MODEL = 'gemini-3.5-flash-lite'
PROMPT_VERSION = 'v1'
BATCH_SIZE = 20
CACHE_PATH = 'cache/llm_labels.jsonl'

PROMPT = open(f'prompts/{PROMPT_VERSION}.txt', encoding='utf-8').read()

def load_cache():
    cache = {}
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, encoding='utf-8') as f:
            for line in f:
                rec = json.loads(line)
                cache[(rec['ticket_id'], rec['prompt_version'])] = rec
    return cache

cache = load_cache()
len(cache)

120

In [8]:
import time

usage = {'calls': 0, 'in_tokens': 0, 'out_tokens': 0}

def call_llm(client, batch):
    from google.genai import types
    tickets = [{'ticket_id': r.ticket_id, 'customer_message': r.customer_message, 'agent_note': r.agent_notes}
               for r in batch]
    prompt = PROMPT.format(tickets=json.dumps(tickets, ensure_ascii=False, indent=1))
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'),
    )
    usage['calls'] += 1
    usage['in_tokens'] += resp.usage_metadata.prompt_token_count
    usage['out_tokens'] += resp.usage_metadata.candidates_token_count
    return json.loads(resp.text)

def label(rows):
    todo = [r for r in rows.itertuples() if (r.ticket_id, PROMPT_VERSION) not in cache]
    print(f'{len(rows) - len(todo)} cached, {len(todo)} to label')
    if not todo:
        return
    if not os.environ.get('GEMINI_API_KEY'):
        print('no GEMINI_API_KEY -> skipping, using cache only')
        return
    from google import genai
    client = genai.Client()
    valid_ids = {r.ticket_id for r in todo}
    with open(CACHE_PATH, 'a', encoding='utf-8') as f:
        for i in range(0, len(todo), BATCH_SIZE):
            batch = todo[i:i + BATCH_SIZE]
            try:
                out = call_llm(client, batch)
            except Exception as e:
                print('batch', i // BATCH_SIZE, 'failed:', str(e)[:150])
                break   # stop on first error, don't burn quota retrying
            for rec in out:
                if rec.get('ticket_id') not in valid_ids:
                    continue   # ignore anything the model made up
                rec.update(prompt_version=PROMPT_VERSION, model=MODEL)
                cache[(rec['ticket_id'], PROMPT_VERSION)] = rec
                f.write(json.dumps(rec) + '\n')
            print(f'batch {i // BATCH_SIZE}: {len(out)} back')
            time.sleep(5)
    print(usage)

In [9]:
# gold set first - no point running 991 rows before we know the accuracy
label(refunds[refunds.ticket_id.isin(gold.ticket_id)])

60 cached, 0 to label


## 4. Score on gold set

Runs once `my_code` is filled in eval/gold_set.csv.

In [10]:
# labels are done in eval/gold_set_labelling.xlsx (dropdowns), copied back to the csv here
XLSX = 'eval/gold_set_labelling.xlsx'
if os.path.exists(XLSX):
    labels = pd.read_excel(XLSX, sheet_name='label_me', dtype=str).fillna('')
    gold = (pd.read_csv(GOLD_PATH, keep_default_na=False)
              .drop(columns=['my_code', 'my_real_goodwill', 'my_comment'], errors='ignore'))
    gold = gold.merge(labels[['ticket_id', 'my_code', 'my_real_goodwill', 'my_comment']], on='ticket_id', how='left')
    gold.to_csv(GOLD_PATH, index=False)

gold = pd.read_csv(GOLD_PATH, keep_default_na=False)
done = gold[gold.my_code != ''].copy()

if len(done) == 0:
    print('gold set not labelled yet')
else:
    done['kw_code'] = [keyword_code(n, m) for n, m in zip(done.agent_notes, done.customer_message)]
    done['llm_code'] = [cache.get((t, PROMPT_VERSION), {}).get('code', '') for t in done.ticket_id]

    scores = pd.DataFrame({
        'agent (original)': (done.refund_reason_code == done.my_code),
        'keyword': (done.kw_code == done.my_code),
        f'llm {PROMPT_VERSION}': (done.llm_code == done.my_code),
    })
    print('n =', len(done))
    print(scores.mean().round(3))
    print(scores.groupby(done.stratum).mean().round(2))

n = 60
agent (original)    0.317
keyword             0.383
llm v1              0.650
dtype: float64
                 agent (original)  keyword  llm v1
stratum                                           
gw_over_cap                  0.16     0.48    0.64
gw_under_cap                 0.30     0.20    0.80
other_code                   0.67     0.53    0.67
refund_and_repl              0.20     0.10    0.50


In [11]:
# where the LLM disagrees with me - this is the 'kind of case it gets wrong' list
if len(done):
    wrong = done[done.llm_code != done.my_code]
    print(len(wrong), 'wrong')
    print(pd.crosstab(wrong.my_code, wrong.llm_code))
    display(wrong[['ticket_id', 'my_code', 'llm_code', 'kw_code', 'my_comment']])

21 wrong
llm_code      DOA-REPL  GW-OTHER  RETURN-QC-OK
my_code                                       
DUP-PAYMENT          0         1             0
GW-OTHER            10         0             1
LOST-TRANSIT         2         0             0
PRICE-ADJ            0         2             0
RETURN-QC-OK         1         0             0
UNCLEAR              0         1             3


,ticket_id,my_code,llm_code,kw_code,my_comment
1,TK-251146,GW-OTHER,DOA-REPL,UNCLEAR,fault -> replacement is the remedy; refund given to avoid escalation. note says rfnd+rplc but flag=N
6,TK-247114,GW-OTHER,DOA-REPL,UNCLEAR,"refund + replacement both, flag=N"
7,TK-249811,LOST-TRANSIT,DOA-REPL,UNCLEAR,"transit damage; note says replacement dispatched but refund also recorded, flag=N"
9,TK-251042,LOST-TRANSIT,DOA-REPL,RETURN-QC-OK,"damaged in transit; refund + new unit, flag=N"
12,TK-253929,DUP-PAYMENT,GW-OTHER,DUP-PAYMENT,note says order manually created -> refund may not be due
13,TK-254433,GW-OTHER,DOA-REPL,UNCLEAR,"reset fixed it, still refund + new unit, flag=N"
18,TK-245488,GW-OTHER,DOA-REPL,UNCLEAR,"fw update resolved, still full refund + RMA new set, flag=N"
20,TK-241686,UNCLEAR,RETURN-QC-OK,LOST-TRANSIT,"refund delay ticket, original refund reason not stated"
21,TK-244029,UNCLEAR,RETURN-QC-OK,UNCLEAR,"refund delay ticket, original refund reason not stated"
25,TK-245407,GW-OTHER,DOA-REPL,UNCLEAR,wrong item shipped - no reason code exists for this


## 5. Prompt v2

v1 mistakes: calls every faulty product DOA-REPL (10 of 21 errors), no UNCLEAR option, doesn't know transit damage = LOST-TRANSIT.

v2 adds: the policy definitions of each code, the "never refund + replacement" rule, shorthand glossary, made-up examples, an UNCLEAR option, confidence and an evidence quote. Also asks `replacement_also_given` because I noticed the agents' replacement flag misses a lot.

Rules come from the policy pdf, examples are invented (not from the gold set) - trying not to just fit these 60.

In [12]:
PROMPT_VERSION = 'v2'
PROMPT = open(f'prompts/{PROMPT_VERSION}.txt', encoding='utf-8').read()
usage = {'calls': 0, 'in_tokens': 0, 'out_tokens': 0}

label(refunds[refunds.ticket_id.isin(gold.ticket_id)])

60 cached, 0 to label


### v2 scores

In [13]:
labels = pd.read_excel(XLSX, sheet_name='label_me', dtype=str).fillna('')
done = done.drop(columns=['my_repl_given'], errors='ignore').merge(labels[['ticket_id', 'my_repl_given']], on='ticket_id')

v2 = {t: cache.get((t, 'v2'), {}) for t in done.ticket_id}
done['v2_code'] = [v2[t].get('code', '') for t in done.ticket_id]
done['v2_repl'] = ['Y' if v2[t].get('replacement_also_given') else 'N' for t in done.ticket_id]
done['v2_conf'] = [v2[t].get('confidence', '') for t in done.ticket_id]

scores = pd.DataFrame({
    'agent (original)': done.refund_reason_code == done.my_code,
    'keyword': done.kw_code == done.my_code,
    'llm v1': done.llm_code == done.my_code,
    'llm v2': done.v2_code == done.my_code,
})
print('reason code accuracy, n =', len(done))
print(scores.mean().round(3), '\n')
print(scores.groupby(done.stratum).mean().round(2), '\n')

# accuracy by the model's own confidence - can we trust 'high'?
print(scores['llm v2'].groupby(done.v2_conf).agg(['mean', 'count']).round(2))

reason code accuracy, n = 60
agent (original)    0.317
keyword             0.383
llm v1              0.650
llm v2              0.867
dtype: float64 

                 agent (original)  keyword  llm v1  llm v2
stratum                                                   
gw_over_cap                  0.16     0.48    0.64    0.96
gw_under_cap                 0.30     0.20    0.80    0.80
other_code                   0.67     0.53    0.67    0.80
refund_and_repl              0.20     0.10    0.50    0.80 

         mean  count
v2_conf             
high     0.87     54
low      0.50      2
medium   1.00      4


In [14]:
# refund + replacement: agent's Y/N flag vs the LLM reading the note, against my reading
for name, col in [('agent flag', 'replacement_issued'), ('llm v2', 'v2_repl')]:
    tp = ((done[col] == 'Y') & (done.my_repl_given == 'Y')).sum()
    fp = ((done[col] == 'Y') & (done.my_repl_given == 'N')).sum()
    fn = ((done[col] == 'N') & (done.my_repl_given == 'Y')).sum()
    print(f'{name:11s} found {tp}/{tp + fn} double payouts, {fp} false alarms')

wrong2 = done[done.v2_code != done.my_code]
print('\nv2 wrong:', len(wrong2))
wrong2[['ticket_id', 'my_code', 'v2_code', 'v2_conf', 'my_comment']]

agent flag  found 10/17 double payouts, 0 false alarms
llm v2      found 17/17 double payouts, 0 false alarms

v2 wrong: 8


,ticket_id,my_code,v2_code,v2_conf,my_comment
11,TK-252082,WTY-BUYBACK,GW-OTHER,high,"faulty unit refunded, could be DOA - purchase date vs delivery unclear"
28,TK-246910,PRICE-ADJ,CANCEL,low,"note useless, reason from customer message"
29,TK-245509,DUP-PAYMENT,UNCLEAR,high,"very thin - payment discrepancy, Rs 32"
45,TK-252397,PRICE-ADJ,UNCLEAR,high,"note says advised coupon terms, resolved - but Rs 6648 refunded. suspicious"
47,TK-240751,RETURN-QC-OK,GW-OTHER,high,"return refund + new unit sent too, flag=N"
48,TK-240529,GW-OTHER,RETURN-QC-OK,high,"Rs 150 for missed pickup - looks like compensation, not a product refund"
53,TK-245423,RETURN-QC-OK,GW-OTHER,high,"note says Rs 332, field says 3324; also new unit sent"
55,TK-243085,LOST-TRANSIT,GW-OTHER,high,"not delivered, full credit + new unit"


## 6. Fresh holdout (20 tickets)

v2 was written after seeing v1's mistakes on the gold set, so 87% might be flattering. 20 new refunds (12 GW-OTHER, 8 other), labelled before running the model.

In [15]:
hold = pd.read_csv('eval/holdout.csv', keep_default_na=False)
label(refunds[refunds.ticket_id.isin(hold.ticket_id)])

hv = {t: cache.get((t, 'v2'), {}) for t in hold.ticket_id}
hold['v2_code'] = [hv[t].get('code', '') for t in hold.ticket_id]
hold['v2_repl'] = ['Y' if hv[t].get('replacement_also_given') else 'N' for t in hold.ticket_id]
hold['kw_code'] = [keyword_code(n, m) for n, m in zip(hold.agent_notes, hold.customer_message)]

print('holdout n =', len(hold))
print('agent code :', (hold.refund_reason_code == hold.my_code).mean().round(2))
print('keyword    :', (hold.kw_code == hold.my_code).mean().round(2))
print('llm v2     :', (hold.v2_code == hold.my_code).mean().round(2))
print('replacement found by llm:', ((hold.v2_repl == 'Y') & (hold.my_repl_given == 'Y')).sum(), '/', (hold.my_repl_given == 'Y').sum(),
      '| false alarms:', ((hold.v2_repl == 'Y') & (hold.my_repl_given == 'N')).sum())
hold.loc[hold.v2_code != hold.my_code, ['ticket_id', 'my_code', 'v2_code', 'agent_notes']]

0 cached, 20 to label


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


batch 0: 20 back


{'calls': 1, 'in_tokens': 2828, 'out_tokens': 1291}
holdout n = 20
agent code : 0.35
keyword    : 0.65
llm v2     : 0.9
replacement found by llm: 3 / 3 | false alarms: 0


,ticket_id,my_code,v2_code,agent_notes
8,TK-249373,RETURN-QC-OK,LOST-TRANSIT,Refund of Rs 2374 initiated to source. Issue: pkp missed. Re-raised pkp w/ courier.
17,TK-249447,GW-OTHER,LOST-TRANSIT,Ticket raised for pkp missed. Re-raised pkp with courier. Refund of Rs 630 initiated to source. //AIS


## 7. Full run with v2

Who gets sent:
- all GW-OTHER refunds (991) - the reason-code question
- other refunds whose note mentions a replacement / new unit / RMA (~260) - to count double payouts the agent flag missed

Everything else keeps the agent's code. ~1,250 tickets = ~63 calls, free tier is 500/day and 15/min.

In [16]:
REPL_WORDS = r'rplc|replac|new unit|new set|new pair|fresh pair|\brma\b|re-?ship|sending a new|new one'

is_gw = refunds.refund_reason_code == 'GW-OTHER'
mentions_repl = refunds.agent_notes.str.contains(REPL_WORDS, case=False, na=False)
to_send = refunds[is_gw | mentions_repl]
print('GW-OTHER:', is_gw.sum(), '| other w/ replacement words:', (~is_gw & mentions_repl).sum(), '| total:', len(to_send))

usage = {'calls': 0, 'in_tokens': 0, 'out_tokens': 0}
label(to_send)   # resumes from cache if it stops halfway

GW-OTHER: 991 | other w/ replacement words: 260 | total: 1251
60 cached, 1191 to label


batch 0: 20 back


batch 1: 20 back


batch 2: 20 back


batch 3: 20 back


...

batch 56: 20 back


batch 57: 20 back


batch 58: 20 back


batch 59: 11 back


{'calls': 60, 'in_tokens': 167794, 'out_tokens': 80866}


In [17]:
import numpy as np

v2_all = {t: rec for (t, v), rec in cache.items() if v == 'v2'}
refunds['llm_code'] = refunds.ticket_id.map(lambda t: v2_all.get(t, {}).get('code'))
refunds['llm_repl'] = refunds.ticket_id.map(lambda t: v2_all.get(t, {}).get('replacement_also_given'))
refunds['llm_evidence'] = refunds.ticket_id.map(lambda t: v2_all.get(t, {}).get('evidence'))

# final reason: AI suggestion only where the agent picked GW-OTHER, agent's code otherwise
refunds['final_code'] = np.where(is_gw & refunds.llm_code.notna(), refunds.llm_code, refunds.refund_reason_code)
refunds['double_payout'] = (refunds.replacement_issued == 'Y') | (refunds.llm_repl == True)

print('GW-OTHER sent but not labelled (rerun to finish):', (is_gw & refunds.llm_code.isna()).sum())

# what the 991 GW-OTHER refunds really are
refunds[is_gw].groupby('final_code').refund_inr.agg(['count', 'sum']).sort_values('sum', ascending=False)

GW-OTHER sent but not labelled (rerun to finish): 1


,count,sum
final_code,,
GW-OTHER,203,712619.0
RETURN-QC-OK,206,595301.0
DUP-PAYMENT,177,460276.0
LOST-TRANSIT,136,406754.0
CANCEL,121,352006.0
UNCLEAR,63,177474.0
WTY-BUYBACK,23,69620.0
DOA-REPL,29,67924.0
PRICE-ADJ,33,65062.0


### Results - what moves money

In [18]:
refunds['quarter'] = pd.PeriodIndex(refunds.refund_month, freq='M').asfreq('Q').astype(str)

# goodwill over the Rs 500 cap: still goodwill after AI reading, or mis-coded?
over = refunds[refunds.flag_gw_over_cap].copy()
over['kind'] = np.select([over.final_code == 'GW-OTHER', over.final_code == 'UNCLEAR'],
                         ['real goodwill over cap', 'unclear'], 'mis-coded (real reason found)')
print(over.groupby('kind').refund_inr.agg(['count', 'sum']), '\n')

real_gw = over[over.kind == 'real goodwill over cap']
print('excess above the Rs 500 cap on real goodwill, per quarter:')
print((real_gw.refund_inr - 500).groupby(real_gw.quarter).sum().astype(int))

                               count        sum
kind                                           
mis-coded (real reason found)    628  1985759.0
real goodwill over cap           196   710453.0
unclear                           55   175420.0 

excess above the Rs 500 cap on real goodwill, per quarter:
quarter
2025Q1     33882
2025Q2     55217
2025Q3     70649
2025Q4    199643
2026Q1    138112
2026Q2    108951
2026Q3      5999
Name: refund_inr, dtype: int64


In [19]:
# double payouts: agent flag only vs flag + AI reading the note
repl_cost = refunds.unit_cost_inr + 340
refunds['dp_leak'] = np.where(refunds.double_payout, np.minimum(refunds.refund_inr, repl_cost), 0)   # conservative

flag_only = refunds.replacement_issued == 'Y'
print('double payouts - agent flag:', flag_only.sum(), '| flag + AI:', refunds.double_payout.sum(),
      '| found only by AI:', (refunds.double_payout & ~flag_only).sum(), '\n')

dp = refunds[refunds.double_payout]
pd.DataFrame({
    'count': dp.groupby('quarter').size(),
    'leak_conservative': dp.groupby('quarter').dp_leak.sum().astype(int),
    'by_team_top': dp.groupby('quarter').agent_team.agg(lambda s: s.value_counts().index[0]),
})

double payouts - agent flag: 166 | flag + AI: 365 | found only by AI: 199 



,count,leak_conservative,by_team_top
quarter,,,
2025Q1,28,49694,Logistics
2025Q2,37,63400,Logistics
2025Q3,60,108415,Logistics
2025Q4,78,158157,Chat Frontline
2026Q1,83,138711,Chat Frontline
2026Q2,78,141705,Chat Frontline
2026Q3,1,2790,Escalations & Warranty


In [20]:
cols = ['ticket_id', 'refund_month', 'quarter', 'agent_id', 'agent_name', 'agent_team', 'product_sku',
        'refund_inr', 'refund_reason_code', 'final_code', 'llm_code', 'llm_evidence',
        'replacement_issued', 'llm_repl', 'double_payout', 'dp_leak', 'flag_gw_over_cap']
refunds[cols].to_csv('output/refunds_labelled.csv', index=False)
print('saved', len(refunds), '| total still reconciles:', round(refunds.refund_inr.sum()))

saved 2340 | total still reconciles: 6709932


## Notes

- Gold set: 60 refunds, stratified. Labelled before any model output existed.
- Reason code accuracy on the 60: agent's own code 32%, keyword rules 38%, **LLM v1 65%, LLM v2 87%**.
- v1 -> v2: biggest fix was the DOA-REPL mistake (any faulty product -> DOA). Policy definitions + "fault fixed or replaced = not DOA" + UNCLEAR option fixed most of it.
- **Fresh holdout (20, labelled before running):** v2 90%, agent 35%, keyword 65%. So v2 isn't just fitted to the gold set.
- **Refund + replacement:** on gold, agent flag caught 10/17, LLM 17/17. Holdout 3/3. 0 false alarms in both.
- v2 still gets wrong: missed-pickup refunds (calls them LOST-TRANSIT), and refund+new-unit on a return/transit case (calls it GW-OTHER). Notes with no content -> UNCLEAR, which is right.
- Confidence is not useful: almost everything comes back 'high'.

**Full run (1,251 refunds, 61 calls, ~168k in / 81k out tokens, free tier):**
- Of 991 GW-OTHER refunds only **203 (Rs 7.1L) are really goodwill**. The rest have a real reason in the notes.
- Of the 879 over the Rs 500 cap: 628 mis-coded (Rs 19.9L), **196 real goodwill over the cap (Rs 7.1L)**, 55 unclear. Excess above the cap ~Rs 1.1-2.0L a quarter.
- **Double payouts: 166 by the agent flag -> 365 with the AI reading notes** (199 the flag missed). Conservative leak ~**Rs 1.4L a quarter** in 2026, mostly Chat Frontline now, Logistics before.
- 1 GW-OTHER ticket didn't come back from the model -> rerun picks it up.
- To check next: spot-check ~20 of the 199 AI-only double payouts before putting the number in front of Finance. Also double payouts and over-cap goodwill can overlap - don't add them blindly.